# Cybersecurity Data Preprocessing Workshop

**Goal:** Practice security-specific preprocessing on network/log data: cleaning, feature extraction, anonymization, scaling, handling class imbalance, and simple modeling.

**What you'll do:**
1. Load a synthetic security dataset (logs + network fields)
2. Clean & standardize timestamps, remove invalid rows
3. Anonymize sensitive fields (e.g., IPs, emails)
4. Extract security features (URL heuristics, ports, protocols)
5. Scale features; handle imbalanced labels
6. Train a simple baseline model with class weighting

> Dataset saved as `synthetic_security_logs.csv`. No internet required.


In [ ]:
import pandas as pd
import numpy as np
import re, hashlib
from datetime import datetime
from collections import Counter
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

DATA_PATH = 'synthetic_security_logs.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 1) Cleaning & Timestamp Normalization
- Parse timestamps to a unified timezone
- Remove obviously invalid rows (e.g., `bytes==0`)
- Keep only relevant protocols and ensure ports are in valid range

In [ ]:
# Parse timestamps to UTC-aware datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
df = df.dropna(subset=['timestamp'])

# Remove empty/invalid transfers
df = df[df['bytes'] > 0]

# Keep valid port range
df = df[(df['src_port'] >= 0) & (df['src_port'] <= 65535) &
        (df['dst_port'] >= 0) & (df['dst_port'] <= 65535)]

# Normalize protocol values
df['protocol'] = df['protocol'].str.upper().str.strip()
df = df[df['protocol'].isin(['TCP','UDP','ICMP'])]

print(df.shape)
df.head()

## 2) Privacy: Anonymize Sensitive Fields
- Hash IP addresses (one-way)
- Redact email-like tokens in URLs (simple regex)

> In real settings, choose a policy (prefix-preserving pseudonyms, tokenization service, etc.).

In [ ]:
def sha256_str(s: str) -> str:
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

# Hash IPs
df['src_ip_hash'] = df['src_ip'].astype(str).apply(sha256_str)
df['dst_ip_hash'] = df['dst_ip'].astype(str).apply(sha256_str)

# Redact emails within URL query/path
email_regex = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
def redact_emails(u: str) -> str:
    if not isinstance(u, str):
        return u
    return email_regex.sub('[REDACTED_EMAIL]', u)
df['url_redacted'] = df['url'].astype(str).apply(redact_emails)
df[['src_ip','dst_ip','src_ip_hash','dst_ip_hash','url','url_redacted']].head(8)

## 3) Feature Engineering (Security-Specific)
Heuristics often used in phishing/URL risk scoring and traffic profiling:
- URL length, number of dots, digits, suspicious keywords
- Has IP address in hostname
- TLD category (benign-ish vs uncommon)
- Ports grouped into service categories (web/ssh/mail/dns/other)
- Protocol one-hot
- Bytes (log-transformed)

In [ ]:
sus_words = set(['login','verify','account','secure','update','free','win','bank'])
bad_tlds = set(['.ru','.cn','.top','.xyz'])
ip_in_host_regex = re.compile(r"(\d{1,3}\.){3}\d{1,3}")

def url_features(u: str):
    if not isinstance(u, str):
        return {
            'url_len': 0,
            'url_num_dots': 0,
            'url_num_digits': 0,
            'url_has_ip': 0,
            'url_has_susp_word': 0,
            'url_bad_tld': 0,
            'path_len': 0,
        }
    url = u.strip()
    url_len = len(url)
    url_num_dots = url.count('.')
    url_num_digits = sum(ch.isdigit() for ch in url)
    url_has_ip = 1 if ip_in_host_regex.search(url) else 0
    url_has_susp_word = 1 if any(w in url.lower() for w in sus_words) else 0
    url_bad_tld = 1 if any(url.lower().endswith(t) for t in bad_tlds) else 0
    # crude path length proxy
    parts = url.split('/')
    path_len = len(parts) - 3 if len(parts) > 3 else 0
    return {
        'url_len': url_len,
        'url_num_dots': url_num_dots,
        'url_num_digits': url_num_digits,
        'url_has_ip': url_has_ip,
        'url_has_susp_word': url_has_susp_word,
        'url_bad_tld': url_bad_tld,
        'path_len': path_len,
    }

url_feat_df = df['url_redacted'].apply(url_features).apply(pd.Series)
df = pd.concat([df, url_feat_df], axis=1)

# Port category mapping
def port_category(p):
    if p in [80, 443, 8080]:
        return 'web'
    if p in [22]:
        return 'ssh'
    if p in [25, 587, 465]:
        return 'mail'
    if p in [53]:
        return 'dns'
    if p in [3306, 5432]:
        return 'db'
    return 'other'

df['src_port_cat'] = df['src_port'].apply(port_category)
df['dst_port_cat'] = df['dst_port'].apply(port_category)

# One-hot encode protocol and port categories
proto_ohe = pd.get_dummies(df['protocol'], prefix='proto')
srcp_ohe = pd.get_dummies(df['src_port_cat'], prefix='srcp')
dstp_ohe = pd.get_dummies(df['dst_port_cat'], prefix='dstp')

df['bytes_log1p'] = np.log1p(df['bytes'])

feat_df = pd.concat([
    df[['url_len','url_num_dots','url_num_digits','url_has_ip','url_has_susp_word','url_bad_tld','path_len','bytes_log1p']],
    proto_ohe, srcp_ohe, dstp_ohe
], axis=1).fillna(0)

feat_df.head()

## 4) Class Imbalance & Scaling
- Inspect label distribution
- Use `class_weight='balanced'` for a quick baseline
- Standardize numerical features

In [ ]:
labels = df['label'].astype(int)
print('Label distribution:', Counter(labels))

# Simple bar chart (no custom colors)
counts = pd.Series(labels).value_counts().sort_index()
plt.figure()
counts.plot(kind='bar')
plt.title('Label Distribution (0=benign, 1=malicious)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(feat_df.values, labels.values, test_size=0.25, random_state=42, stratify=labels)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled[:2].shape, X_test_scaled[:2].shape

## 5) Quick Baseline Model (Optional but Educational)
- Logistic Regression with `class_weight='balanced'`
- Evaluate with precision/recall/F1 (recall is important for catching attacks)

In [ ]:
clf = LogisticRegression(max_iter=200, class_weight='balanced', n_jobs=None)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

## 6) Takeaways & Extensions
- Cleaning removes invalid noise and aligns time across sources
- Privacy matters: hash IPs, redact PII
- Security-specific features: URL heuristics, protocol/port categories, byte volume
- Handle imbalance (class weights or sampling)
- Try more models (e.g., tree-based), sequence features (API/syscall sequences), or PCAP sessionization

**Challenge ideas:**
1. Add a rule: mark URLs with >3 dots *and* a suspicious word as high risk
2. Build session-level features (group by 5-tuple: src/dst IP/port + protocol)
3. Replace hashing with prefix-preserving IP anonymization (k-anonymity on subnets)
